In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

# Загружаем данные
df = pd.read_csv('/Users/sofya/Desktop/diplomahse/ozon_train.csv')
print("Строк, колонок:", df.shape)
print("\nДоля контрафакта:")
print(df['resolution'].value_counts(normalize=True))
print("\nФинтех-колонки есть?")
print([c for c in df.columns if any(x in c for x in ['return', 'sales', 'seller', 'time', 'Gmv'])])

Строк, колонок: (197198, 45)

Доля контрафакта:
resolution
0    0.933813
1    0.066187
Name: proportion, dtype: float64

Финтех-колонки есть?
['item_time_alive', 'item_count_fake_returns7', 'item_count_fake_returns30', 'item_count_fake_returns90', 'item_count_sales7', 'item_count_sales30', 'item_count_sales90', 'item_count_returns7', 'item_count_returns30', 'item_count_returns90', 'GmvTotal7', 'GmvTotal30', 'GmvTotal90', 'seller_time_alive']


In [3]:
np.random.seed(42)

seller_class_counts = df.groupby("SellerID")["resolution"].nunique()
multi_class_sellers = seller_class_counts[seller_class_counts > 1].index

test_sellers = np.random.choice(
    multi_class_sellers,
    size=int(0.3 * len(multi_class_sellers)),
    replace=False
)
train_sellers = np.setdiff1d(df["SellerID"].unique(), test_sellers)

train_df = df[df["SellerID"].isin(train_sellers)].copy()
test_df = df[df["SellerID"].isin(test_sellers)].copy()

print("Train:", train_df.shape, "| контрафакт:", train_df['resolution'].mean().round(4))
print("Test:", test_df.shape, "| контрафакт:", test_df['resolution'].mean().round(4))

Train: (177380, 45) | контрафакт: 0.0588
Test: (19818, 45) | контрафакт: 0.1321


In [4]:
def add_fintech_features(df):
    df = df.copy()
    df['return_rate_30'] = df['item_count_returns30'] / (df['item_count_sales30'] + 1)
    df['fake_return_rate_30'] = df['item_count_fake_returns30'] / (df['item_count_sales30'] + 1)
    df['fake_return_rate_90'] = df['item_count_fake_returns90'] / (df['item_count_sales90'] + 1)
    df['seller_velocity'] = df['item_count_sales30'] / (df['seller_time_alive'] + 1)
    df['gmv_per_day'] = df['GmvTotal30'] / (df['item_time_alive'] + 1)
    df['both_new'] = ((df['item_time_alive'] < 30) & (df['seller_time_alive'] < 90)).astype(int)
    return df

train_df = add_fintech_features(train_df)
test_df = add_fintech_features(test_df)

# Графовые агрегаты только по train — без leakage
seller_stats = train_df.groupby('SellerID').agg(
    seller_item_count=('ItemID', 'count'),
    seller_avg_return_rate=('return_rate_30', 'mean'),
    seller_avg_fake_returns=('fake_return_rate_30', 'mean'),
).reset_index()

train_df = train_df.merge(seller_stats, on='SellerID', how='left')
test_df = test_df.merge(seller_stats, on='SellerID', how='left')
test_df[['seller_item_count','seller_avg_return_rate','seller_avg_fake_returns']] = \
    test_df[['seller_item_count','seller_avg_return_rate','seller_avg_fake_returns']].fillna(0)

print("Фич итого:", train_df.shape[1])
print(train_df[['return_rate_30','fake_return_rate_30','seller_velocity','both_new']].describe().round(3))

Фич итого: 54
       return_rate_30  fake_return_rate_30  seller_velocity    both_new
count      177380.000           177380.000       177380.000  177380.000
mean            0.005                0.001            0.008       0.092
std             0.044                0.025            0.151       0.289
min             0.000                0.000            0.000       0.000
25%             0.000                0.000            0.000       0.000
50%             0.000                0.000            0.000       0.000
75%             0.000                0.000            0.000       0.000
max             2.000                2.000           31.240       1.000
